# Epitranscriptome Analysis Pipeline
## Explainable AI for Detecting RNA Changes in Plasma-Treated Skin Cancer Using Nanopore Sequencing

**Author:** Chandana Nagaraju  
**Thesis:** Masterarbeit, Universität Rostock  

This notebook implements the complete analysis pipeline:
1. Extract modification data from BAM files (MM/ML tags)
2. Compute site-level modification stoichiometry
3. Differential modification analysis between conditions
4. Biological feature construction
5. Train explainable ML models (XGBoost / Random Forest)
6. SHAP-based feature interpretation
7. Pathway enrichment analysis

---

## Setup

In [ ]:
import os
import sys
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

# Add scripts directory to path
sys.path.insert(0, os.path.join(os.getcwd(), 'scripts'))
from utils import *

# Plot settings
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100
sns.set_style('whitegrid')
sns.set_palette('Set2')

print('Setup complete.')

In [ ]:
# Load configuration
config = load_config('config.yaml')

# Display sample information
print('Samples:')
for name, info in config['samples'].items():
    print(f"  {name}: condition={info['condition']}, oxygen={info['oxygen']}")
    print(f"    BAM: {info['bam']}")
    print(f"    Exists: {os.path.exists(info['bam'])}")

print(f"\nComparisons:")
for comp in config['comparisons']:
    print(f"  {comp['control']} vs {comp['treatment']}")

print(f"\nModification threshold: {config['thresholds']['modification_probability']}/255 "
      f"= {config['thresholds']['modification_probability']/255:.3f}")
print(f"Min coverage: {config['thresholds']['min_coverage']}")
print(f"FDR threshold: {config['thresholds']['fdr_threshold']}")

---
## Step 1: Extract Modification Information from BAM Files

Parse MM (modification type) and ML (modification probability) tags from Dorado-basecalled BAM files.

**Modification types detected by `inosine_m6A_2OmeA` model:**
- `A+a` → N6-methyladenosine (m6A)
- `A+17596` → Inosine (ChEBI:17596)
- `A+69426` → 2'-O-methyladenosine (ChEBI:69426)

In [ ]:
import pysam

def extract_modifications(bam_path, sample_name, max_reads=None):
    """Extract modification calls from a BAM file."""
    records = []
    n_reads = 0
    n_with_mods = 0
    
    with pysam.AlignmentFile(bam_path, 'rb', check_sq=False) as bam:
        for read in tqdm(bam.fetch(until_eof=True), desc=sample_name):
            n_reads += 1
            if max_reads and n_reads > max_reads:
                break
            
            if read.is_unmapped or not read.has_tag('MM') or not read.has_tag('ML'):
                continue
            
            n_with_mods += 1
            read_id = read.query_name
            ref_name = read.reference_name
            strand = '-' if read.is_reverse else '+'
            
            # Try pysam native method
            try:
                mod_bases = read.modified_bases
                if mod_bases:
                    aligned_pairs = dict(
                        (q, r) for q, r in read.get_aligned_pairs()
                        if q is not None and r is not None
                    )
                    for (base, st, mod_code), positions in mod_bases.items():
                        mod_name = mod_code_to_name(str(mod_code))
                        for query_pos, quality in positions:
                            ref_pos = aligned_pairs.get(query_pos)
                            if ref_pos is not None:
                                records.append({
                                    'sample': sample_name,
                                    'read_id': read_id,
                                    'reference_name': ref_name,
                                    'ref_pos': ref_pos,
                                    'mod_type': mod_name,
                                    'probability': prob_to_float(quality),
                                    'strand': strand
                                })
                    continue
            except (AttributeError, TypeError):
                pass
            
            # Manual fallback
            mm_string = read.get_tag('MM')
            ml_array = list(read.get_tag('ML'))
            mm_entries = parse_mm_tag(mm_string)
            mod_positions = get_modification_positions(read, mm_entries)
            mod_positions = assign_ml_probabilities(mod_positions, ml_array, mm_entries)
            
            for mp in mod_positions:
                records.append({
                    'sample': sample_name,
                    'read_id': read_id,
                    'reference_name': ref_name,
                    'ref_pos': mp['ref_pos'],
                    'mod_type': mod_code_to_name(mp['mod_code']),
                    'probability': mp.get('probability', 0.0),
                    'strand': strand
                })
    
    print(f"  {sample_name}: {n_reads:,} reads, {n_with_mods:,} with mods, {len(records):,} calls")
    return pd.DataFrame(records)

In [ ]:
# Extract modifications from all samples
# NOTE: Set max_reads=100000 for a quick test run, None for full analysis
MAX_READS = None  # Change to e.g. 100000 for testing

extraction_results = {}
output_dir = config['output']['extraction_dir']

for sample_name, sample_info in config['samples'].items():
    bam_path = sample_info['bam']
    
    # Check if already extracted
    cached_path = os.path.join(output_dir, f"{sample_name}_modifications.parquet")
    if os.path.exists(cached_path):
        print(f"Loading cached extraction for {sample_name}...")
        extraction_results[sample_name] = pd.read_parquet(cached_path)
        print(f"  Loaded {len(extraction_results[sample_name]):,} records")
        continue
    
    if not os.path.exists(bam_path):
        print(f"WARNING: BAM file not found: {bam_path}")
        continue
    
    df = extract_modifications(bam_path, sample_name, max_reads=MAX_READS)
    extraction_results[sample_name] = df
    
    # Cache results
    save_dataframe(df, os.path.join(output_dir, f"{sample_name}_modifications.csv"))
    print(f"  Saved to {output_dir}")

In [ ]:
# Visualize extraction results
for sample_name, df in extraction_results.items():
    print(f"\n=== {sample_name} ===")
    print(f"Total modification calls: {len(df):,}")
    print(f"Unique reads: {df['read_id'].nunique():,}")
    print(f"Unique transcripts: {df['reference_name'].nunique():,}")
    print(f"\nModification type distribution:")
    print(df['mod_type'].value_counts())
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    # Probability distribution per mod type
    for mod_type in df['mod_type'].unique():
        subset = df[df['mod_type'] == mod_type]
        axes[0].hist(subset['probability'], bins=50, alpha=0.6, label=mod_type)
    axes[0].set_xlabel('Modification Probability')
    axes[0].set_ylabel('Count')
    axes[0].set_title(f'{sample_name} - Probability Distribution')
    axes[0].legend()
    
    # Mod type counts
    df['mod_type'].value_counts().plot(kind='bar', ax=axes[1])
    axes[1].set_title('Calls per Modification Type')
    axes[1].set_ylabel('Count')
    
    # Reads per transcript (top 20)
    top_transcripts = df['reference_name'].value_counts().head(20)
    top_transcripts.plot(kind='barh', ax=axes[2])
    axes[2].set_title('Top 20 Transcripts by Mod Calls')
    axes[2].set_xlabel('Modification Calls')
    
    plt.tight_layout()
    plt.savefig(os.path.join(config['output']['figures_dir'], 
                             f'{sample_name}_extraction_overview.png'), dpi=150)
    plt.show()

---
## Step 2: Site-Level Modification Stoichiometry

For each genomic position covered by multiple reads, compute:
- **Stoichiometry** = modified reads / total reads
- A read is classified as **modified** if probability ≥ threshold

In [ ]:
threshold = config['thresholds']['modification_probability'] / 255.0
min_coverage = config['thresholds']['min_coverage']
print(f"Modification probability threshold: {threshold:.3f}")
print(f"Minimum coverage: {min_coverage}x")

stoichiometry_results = {}
stoich_dir = config['output']['stoichiometry_dir']

for sample_name, df in extraction_results.items():
    print(f"\nProcessing {sample_name}...")
    
    # Classify reads as modified/unmodified
    df['is_modified'] = df['probability'] >= threshold
    n_mod = df['is_modified'].sum()
    print(f"  Modified calls: {n_mod:,} / {len(df):,} ({n_mod/len(df)*100:.1f}%)")
    
    # Aggregate to site level
    group_cols = ['sample', 'reference_name', 'ref_pos', 'mod_type', 'strand']
    site_df = df.groupby(group_cols).agg(
        total_reads=('read_id', 'nunique'),
        modified_reads=('is_modified', 'sum'),
        mean_probability=('probability', 'mean'),
        median_probability=('probability', 'median'),
        std_probability=('probability', 'std'),
    ).reset_index()
    
    site_df['stoichiometry'] = site_df['modified_reads'] / site_df['total_reads']
    
    # Coverage filter
    before = len(site_df)
    site_df = site_df[site_df['total_reads'] >= min_coverage].copy()
    print(f"  Sites: {before:,} -> {len(site_df):,} (after {min_coverage}x filter)")
    
    stoichiometry_results[sample_name] = site_df
    save_dataframe(site_df, os.path.join(stoich_dir, f"{sample_name}_stoichiometry.csv"))
    
    # Per mod-type summary
    for mt in site_df['mod_type'].unique():
        sub = site_df[site_df['mod_type'] == mt]
        print(f"  {mt}: {len(sub)} sites, mean stoich={sub['stoichiometry'].mean():.4f}")

In [ ]:
# Visualize stoichiometry
fig, axes = plt.subplots(1, len(stoichiometry_results), figsize=(7*len(stoichiometry_results), 5))
if len(stoichiometry_results) == 1:
    axes = [axes]

for idx, (sample_name, site_df) in enumerate(stoichiometry_results.items()):
    ax = axes[idx]
    for mt in sorted(site_df['mod_type'].unique()):
        sub = site_df[site_df['mod_type'] == mt]
        ax.hist(sub['stoichiometry'], bins=50, alpha=0.6, label=f"{mt} (n={len(sub)})")
    ax.set_xlabel('Stoichiometry')
    ax.set_ylabel('Number of Sites')
    ax.set_title(f'{sample_name}')
    ax.legend()

plt.suptitle('Site-Level Modification Stoichiometry Distribution', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(config['output']['figures_dir'], 'stoichiometry_distributions.png'), dpi=150)
plt.show()

---
## Step 3: Differential Modification Analysis

Compare modification stoichiometry between conditions using Fisher's exact test.

For each site present in both conditions:
- H₀: No difference in modification frequency between conditions
- Δ stoichiometry = stoichiometry_treatment − stoichiometry_control
- Benjamini-Hochberg FDR correction

In [ ]:
from statsmodels.stats.multitest import multipletests

diff_dir = config['output']['differential_dir']
fdr_threshold = config['thresholds']['fdr_threshold']
min_delta = config['thresholds']['min_delta_stoichiometry']

all_differential = []

for comp in config['comparisons']:
    ctrl_name = comp['control']
    treat_name = comp['treatment']
    comp_name = f"{ctrl_name}_vs_{treat_name}"
    
    print(f"\n=== {comp_name} ===")
    
    if ctrl_name not in stoichiometry_results or treat_name not in stoichiometry_results:
        print(f"  Missing data for one or both conditions. Skipping.")
        continue
    
    ctrl_df = stoichiometry_results[ctrl_name]
    treat_df = stoichiometry_results[treat_name]
    
    # Merge on site identity
    merge_cols = ['reference_name', 'ref_pos', 'mod_type', 'strand']
    merged = ctrl_df.merge(treat_df, on=merge_cols, suffixes=('_ctrl', '_treat'), how='inner')
    print(f"  Control sites: {len(ctrl_df):,}")
    print(f"  Treatment sites: {len(treat_df):,}")
    print(f"  Common sites: {len(merged):,}")
    
    for mod_type in sorted(merged['mod_type'].unique()):
        print(f"\n  --- {mod_type} ---")
        subset = merged[merged['mod_type'] == mod_type].copy()
        
        if len(subset) < 2:
            print(f"  Too few sites. Skipping.")
            continue
        
        # Fisher's exact test for each site
        pvalues = []
        for _, row in tqdm(subset.iterrows(), total=len(subset), desc='  Testing'):
            mod_c = int(row['modified_reads_ctrl'])
            unmod_c = int(row['total_reads_ctrl'] - row['modified_reads_ctrl'])
            mod_t = int(row['modified_reads_treat'])
            unmod_t = int(row['total_reads_treat'] - row['modified_reads_treat'])
            
            try:
                _, pval = stats.fisher_exact([[mod_c, unmod_c], [mod_t, unmod_t]])
            except Exception:
                pval = 1.0
            pvalues.append(pval)
        
        subset['pvalue'] = pvalues
        subset['delta_stoichiometry'] = subset['stoichiometry_treat'] - subset['stoichiometry_ctrl']
        subset['abs_delta_stoichiometry'] = subset['delta_stoichiometry'].abs()
        
        # Multiple testing correction
        reject, fdr, _, _ = multipletests(subset['pvalue'], method='fdr_bh', alpha=0.05)
        subset['fdr'] = fdr
        subset['is_significant'] = (subset['fdr'] < fdr_threshold) & (subset['abs_delta_stoichiometry'] >= min_delta)
        subset['direction'] = 'unchanged'
        subset.loc[subset['is_significant'] & (subset['delta_stoichiometry'] > 0), 'direction'] = 'increased'
        subset.loc[subset['is_significant'] & (subset['delta_stoichiometry'] < 0), 'direction'] = 'decreased'
        
        subset['control'] = ctrl_name
        subset['treatment'] = treat_name
        
        n_sig = subset['is_significant'].sum()
        n_up = (subset['direction'] == 'increased').sum()
        n_down = (subset['direction'] == 'decreased').sum()
        
        print(f"  Tested: {len(subset):,}")
        print(f"  Significant (FDR<{fdr_threshold}, |Δ|≥{min_delta}): {n_sig}")
        print(f"    Increased: {n_up}, Decreased: {n_down}")
        
        save_dataframe(subset, os.path.join(diff_dir, f"{comp_name}_{mod_type}_differential.csv"))
        all_differential.append(subset)

if all_differential:
    all_diff_df = pd.concat(all_differential, ignore_index=True)
    save_dataframe(all_diff_df, os.path.join(diff_dir, 'all_differential_results.csv'))
    print(f"\nTotal differential results: {len(all_diff_df):,}")

In [ ]:
# Volcano plot
if all_differential:
    fig, ax = plt.subplots(figsize=(10, 7))
    
    df_plot = all_diff_df.copy()
    df_plot['-log10_fdr'] = -np.log10(df_plot['fdr'].clip(lower=1e-300))
    
    colors = {'unchanged': '#cccccc', 'increased': '#e74c3c', 'decreased': '#3498db'}
    
    for direction, color in colors.items():
        mask = df_plot['direction'] == direction
        ax.scatter(
            df_plot.loc[mask, 'delta_stoichiometry'],
            df_plot.loc[mask, '-log10_fdr'],
            c=color, alpha=0.5, s=10, label=direction
        )
    
    ax.axhline(-np.log10(fdr_threshold), color='grey', linestyle='--', alpha=0.5)
    ax.axvline(min_delta, color='grey', linestyle='--', alpha=0.5)
    ax.axvline(-min_delta, color='grey', linestyle='--', alpha=0.5)
    
    ax.set_xlabel('Δ Stoichiometry (Treatment − Control)', fontsize=12)
    ax.set_ylabel('-log₁₀(FDR)', fontsize=12)
    ax.set_title('Volcano Plot: Differential RNA Modifications', fontsize=14)
    ax.legend(fontsize=11)
    
    plt.tight_layout()
    plt.savefig(os.path.join(config['output']['figures_dir'], 'volcano_plot.png'), dpi=150)
    plt.show()
else:
    print('No differential results to plot.')

---
## Step 4: Biological Feature Construction

Build feature vectors for each modification site incorporating:
- Baseline stoichiometry
- Transcript region (5'UTR / CDS / 3'UTR)
- GC content
- DRACH motif presence
- Gene length
- Coverage metrics
- Pathway annotations

In [ ]:
from scripts.utils import load_dataframe
from scripts import _04_feature_construction_module as feat_module

# If the import above fails, use the functions directly:
try:
    from scripts._04_feature_construction_module import build_feature_matrix, prepare_ml_features
except ImportError:
    # Inline the feature construction
    exec(open('scripts/04_feature_construction.py').read().split('def main')[0])

print('Feature construction functions loaded.')

In [ ]:
# Build feature matrix from differential results
import logging
logger = logging.getLogger('features')
logger.setLevel(logging.INFO)
if not logger.handlers:
    logger.addHandler(logging.StreamHandler())

if all_differential:
    feature_df = build_feature_matrix(all_diff_df.copy(), config, logger)
    feature_df, feature_cols = prepare_ml_features(feature_df, logger)
    feature_df['label'] = feature_df['is_significant'].astype(int)
    
    features_dir = config['output']['features_dir']
    save_dataframe(feature_df, os.path.join(features_dir, 'feature_matrix.csv'))
    
    with open(os.path.join(features_dir, 'feature_columns.txt'), 'w') as f:
        f.write('\n'.join(feature_cols))
    
    print(f"Feature matrix: {feature_df.shape}")
    print(f"Features: {feature_cols}")
    print(f"Positive (significant): {feature_df['label'].sum()}")
    print(f"Negative: {(feature_df['label'] == 0).sum()}")
else:
    print('No differential results available for feature construction.')

---
## Step 5: Train Explainable ML Models

Train XGBoost and Random Forest classifiers to predict which sites are susceptible to modification change.

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import classification_report, roc_auc_score, roc_curve, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
import pickle

if 'feature_df' not in dir() or feature_df is None:
    print('Feature matrix not available. Run Step 4 first.')
else:
    X = feature_df[feature_cols]
    y = feature_df['label']
    
    if y.nunique() < 2:
        print("WARNING: Only one class present. Cannot train classifier.")
        print("This is expected when comparing similar conditions (normoxia vs hypoxia).")
        print("The model will work properly once plasma-treated data is available.")
    else:
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42, stratify=y
        )
        
        print(f"Train: {len(X_train)} samples ({y_train.sum()} positive)")
        print(f"Test:  {len(X_test)} samples ({y_test.sum()} positive)")
        
        # --- XGBoost ---
        try:
            import xgboost as xgb
            
            scale_pos = (y_train == 0).sum() / max(y_train.sum(), 1)
            xgb_model = xgb.XGBClassifier(
                n_estimators=200, max_depth=6, learning_rate=0.1,
                scale_pos_weight=scale_pos, eval_metric='logloss',
                use_label_encoder=False, random_state=42
            )
            xgb_model.fit(X_train, y_train)
            
            y_pred = xgb_model.predict(X_test)
            y_proba = xgb_model.predict_proba(X_test)[:, 1]
            
            print("\n=== XGBoost Results ===")
            print(classification_report(y_test, y_pred))
            try:
                print(f"ROC AUC: {roc_auc_score(y_test, y_proba):.4f}")
            except ValueError:
                pass
            
            # Save model
            model_dir = config['output']['model_dir']
            with open(os.path.join(model_dir, 'xgboost_model.pkl'), 'wb') as f:
                pickle.dump(xgb_model, f)
        except ImportError:
            print("XGBoost not installed. Skipping.")
            xgb_model = None
        
        # --- Random Forest ---
        rf_model = RandomForestClassifier(
            n_estimators=200, max_depth=6, class_weight='balanced',
            random_state=42, n_jobs=-1
        )
        rf_model.fit(X_train, y_train)
        
        y_pred_rf = rf_model.predict(X_test)
        y_proba_rf = rf_model.predict_proba(X_test)[:, 1]
        
        print("\n=== Random Forest Results ===")
        print(classification_report(y_test, y_pred_rf))
        try:
            print(f"ROC AUC: {roc_auc_score(y_test, y_proba_rf):.4f}")
        except ValueError:
            pass
        
        with open(os.path.join(model_dir, 'random_forest_model.pkl'), 'wb') as f:
            pickle.dump(rf_model, f)
        
        # Save test data
        test_data = pd.DataFrame(X_test, columns=feature_cols)
        test_data['y_true'] = y_test.values
        test_data.to_csv(os.path.join(model_dir, 'test_data.csv'), index=False)

In [ ]:
# ROC curves (if models were trained)
if 'xgb_model' in dir() and xgb_model is not None and y.nunique() >= 2:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    try:
        fpr_xgb, tpr_xgb, _ = roc_curve(y_test, y_proba)
        auc_xgb = roc_auc_score(y_test, y_proba)
        axes[0].plot(fpr_xgb, tpr_xgb, label=f'XGBoost (AUC={auc_xgb:.3f})')
    except ValueError:
        pass
    
    try:
        fpr_rf, tpr_rf, _ = roc_curve(y_test, y_proba_rf)
        auc_rf = roc_auc_score(y_test, y_proba_rf)
        axes[0].plot(fpr_rf, tpr_rf, label=f'Random Forest (AUC={auc_rf:.3f})')
    except ValueError:
        pass
    
    axes[0].plot([0,1], [0,1], 'k--', alpha=0.3)
    axes[0].set_xlabel('False Positive Rate')
    axes[0].set_ylabel('True Positive Rate')
    axes[0].set_title('ROC Curve')
    axes[0].legend()
    
    # Feature importance comparison
    imp_xgb = pd.Series(xgb_model.feature_importances_, index=feature_cols).sort_values(ascending=True).tail(15)
    imp_xgb.plot(kind='barh', ax=axes[1])
    axes[1].set_title('XGBoost Feature Importance (Top 15)')
    axes[1].set_xlabel('Importance')
    
    plt.tight_layout()
    plt.savefig(os.path.join(config['output']['figures_dir'], 'model_performance.png'), dpi=150)
    plt.show()

---
## Step 6: SHAP Analysis

Use SHAP (SHapley Additive exPlanations) to interpret the trained model
and identify which biological features contribute most to modification susceptibility.

In [ ]:
if 'xgb_model' in dir() and xgb_model is not None and y.nunique() >= 2:
    import shap
    
    # Use TreeExplainer for fast exact SHAP values
    explainer = shap.TreeExplainer(xgb_model)
    shap_values = explainer.shap_values(X_test)
    
    # If binary classification returns list, take class 1
    if isinstance(shap_values, list):
        shap_values = shap_values[1]
    
    print(f"SHAP values computed: {shap_values.shape}")
    
    # Save SHAP values
    shap_dir = config['output']['shap_dir']
    shap_df = pd.DataFrame(shap_values, columns=feature_cols[:shap_values.shape[1]])
    shap_df.to_csv(os.path.join(shap_dir, 'shap_values.csv'), index=False)
    
    # Global importance
    mean_abs_shap = np.abs(shap_values).mean(axis=0)
    importance_df = pd.DataFrame({
        'feature': feature_cols[:len(mean_abs_shap)],
        'mean_abs_shap': mean_abs_shap
    }).sort_values('mean_abs_shap', ascending=False)
    
    print("\nGlobal SHAP Feature Importance:")
    print(importance_df.to_string(index=False))
    
    importance_df.to_csv(os.path.join(shap_dir, 'shap_global_importance.csv'), index=False)
else:
    print('Model not trained. SHAP analysis requires a trained model with both classes.')

In [ ]:
# SHAP visualizations
if 'shap_values' in dir():
    # Summary bar plot
    fig, ax = plt.subplots(figsize=(10, 8))
    shap.summary_plot(shap_values, X_test, feature_names=feature_cols[:shap_values.shape[1]],
                      plot_type='bar', show=False, max_display=20)
    plt.title('SHAP Feature Importance (Global)', fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(config['output']['figures_dir'], 'shap_summary_bar.png'), dpi=150, bbox_inches='tight')
    plt.show()
    
    # Beeswarm plot
    fig, ax = plt.subplots(figsize=(10, 8))
    shap.summary_plot(shap_values, X_test, feature_names=feature_cols[:shap_values.shape[1]],
                      show=False, max_display=20)
    plt.title('SHAP Beeswarm Plot', fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(config['output']['figures_dir'], 'shap_beeswarm.png'), dpi=150, bbox_inches='tight')
    plt.show()

---
## Step 7: Pathway Enrichment Analysis

Identify biological pathways enriched among genes with significant modification changes.

In [ ]:
from scripts.utils import load_dataframe

# Load feature matrix with gene annotations
features_dir = config['output']['features_dir']
pathway_dir = config['output']['pathway_dir']

if 'feature_df' in dir() and feature_df is not None and 'gene_name' in feature_df.columns:
    sig_genes = feature_df[feature_df['is_significant'] == True]['gene_name'].unique().tolist()
    all_genes = feature_df['gene_name'].unique().tolist()
    
    print(f"Significant genes: {len(sig_genes)}")
    print(f"Background genes: {len(all_genes)}")
    
    if len(sig_genes) == 0:
        print("\nNo significant genes found - using top changed sites for demo")
        top_df = feature_df.nlargest(min(100, len(feature_df)), 'abs_delta_stoichiometry')
        sig_genes = top_df['gene_name'].unique().tolist()
        print(f"Using top {len(sig_genes)} genes by effect size")
else:
    print('Feature matrix not available.')
    sig_genes = []
    all_genes = []

In [ ]:
# Run curated pathway enrichment
if sig_genes:
    # Import from Step 7 script
    exec(open('scripts/07_pathway_enrichment.py').read().split('def main')[0])
    
    curated_results = curated_pathway_enrichment(sig_genes, all_genes, logger)
    
    print("\n=== Curated Pathway Enrichment Results ===")
    if not curated_results.empty:
        display(curated_results[[
            'pathway', 'pathway_size', 'overlap_count', 
            'fold_enrichment', 'pvalue', 'fdr', 'significant', 'overlap_genes'
        ]].round(4))
        
        curated_results.to_csv(os.path.join(pathway_dir, 'curated_pathway_enrichment.csv'), index=False)
    else:
        print('No pathway results generated.')

In [ ]:
# Pathway enrichment visualization
if 'curated_results' in dir() and not curated_results.empty:
    fig, ax = plt.subplots(figsize=(10, 6))
    
    plot_df = curated_results.sort_values('pvalue').head(10)
    plot_df['-log10_pvalue'] = -np.log10(plot_df['pvalue'].clip(lower=1e-300))
    
    colors = ['#e74c3c' if s else '#95a5a6' for s in plot_df['significant']]
    
    bars = ax.barh(plot_df['pathway'], plot_df['-log10_pvalue'], color=colors)
    ax.axvline(-np.log10(0.05), color='grey', linestyle='--', alpha=0.5, label='p=0.05')
    
    # Add overlap counts
    for bar, (_, row) in zip(bars, plot_df.iterrows()):
        ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
                f"{int(row['overlap_count'])}/{int(row['pathway_size'])}",
                va='center', fontsize=9)
    
    ax.set_xlabel('-log₁₀(p-value)', fontsize=12)
    ax.set_title('Pathway Enrichment Analysis', fontsize=14)
    ax.legend()
    
    plt.tight_layout()
    plt.savefig(os.path.join(config['output']['figures_dir'], 'pathway_enrichment.png'), dpi=150)
    plt.show()

---
## Summary & Next Steps

### Current Status
This pipeline has been run with the available untreated data:
- **A375_normoxia** (skin cancer, normal oxygen)
- **A375_hypoxia** (skin cancer, low oxygen)

### When Plasma-Treated Data Becomes Available
1. Update `config.yaml` to add the plasma-treated BAM file paths
2. Add new comparison pairs (e.g., normoxia vs normoxia_plasma)
3. Re-run the pipeline: `bash run_pipeline.sh`

### Expected Biological Insights
With plasma-treated data, the analysis should reveal:
- RNA modification sites significantly altered by plasma treatment
- Enrichment of changes in oxidative stress response genes
- Preferential modification changes in 3'UTR regions and DRACH motifs
- Pathway associations linking epitranscriptomic changes to apoptosis/DNA damage

### Output Files
All results are organized in the `results/` directory.

In [ ]:
# Print output directory structure
import subprocess
result = subprocess.run(['find', 'results', '-type', 'f', '-name', '*.csv', '-o', '-name', '*.png', '-o', '-name', '*.pkl'],
                       capture_output=True, text=True)
print('Generated output files:')
for line in sorted(result.stdout.strip().split('\n')):
    if line:
        print(f'  {line}')